In [1]:
from pymongo import MongoClient
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import pandas as pd
pd.set_option('display.max_colwidth', None)


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cpu
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:


def create_laptop_document(row):
    parts = []

    name = row.get("name_product")
    brand = row.get("brand")
    if pd.notnull(name) and pd.notnull(brand):
        parts.append(f"Laptop {name} thương hiệu {brand}.")
    
    price = row.get("price")
    if pd.notnull(price) and price > 0:
        parts.append(f"Giá bán: {price/1_000_000:.2f} triệu VNĐ.")

    # make cpu, ram, rom, gpu into group to easily manage
    perf = []
    if pd.notnull(row.get("cpu_name")): 
        perf.append(f"Chip xử lí {row.get('cpu_name')}")
    if pd.notnull(row.get("ram_info")): 
        perf.append(f"Bộ nhớ RAM {row.get('ram_info')}GB {row.get('ram_type', '')}")
    if pd.notnull(row.get("max_ram_upgrade")): 
        perf.append(f"Khả năng nâng cấp ram tối đa là {row.get('max_ram_upgrade')} GB")
    if pd.notnull(row.get("gpu_type")): 
        perf.append(f"card đồ họa của máy là card {row.get('gpu_type')}")
    if pd.notnull(row.get("gpu_ram")): 
        perf.append(f" card có bộ nhớ là {row.get('gpu_ram')}GB")
    if pd.notnull(row.get("gpu_name")): 
        perf.append(f"{row.get('gpu_name')}")
    if pd.notnull(row.get("storage_capacity")): 
        perf.append(f"ổ cứng trong có dung lượng là {row.get('storage_capacity')}")
    
    if perf:
        parts.append("Cấu hình laptop gồm: " + ", ".join(perf) + ".")

    # 3. make all infomation about display into group to easily manage
    disp = []
    if pd.notnull(row.get("display_size")): disp.append(f"{row.get('display_size')} inch")
    if pd.notnull(row.get("resolution")): disp.append(row.get("resolution"))
    if pd.notnull(row.get("panel_type")): disp.append(row.get("panel_type"))
    if pd.notnull(row.get("refresh_rate")): disp.append(f"{int(row['refresh_rate'])}Hz")
    if pd.notnull(row.get("color_gamut")): disp.append(f"độ phủ màu {row.get('color_gamut')}")
    if pd.notnull(row.get("webcam")): disp.append(f"Máy được tích hợp Webcam là {row.get('webcam')}.")

    if disp:
        parts.append("Về Phần Màn hình laptop: " + ", ".join(disp) + ".")

    # 4. Nhóm Thiết kế & Năm ra mắt (Theo ý ông muốn thêm vào)
    design = []
    if pd.notnull(row.get("material")): design.append(f"vỏ máy tính được làm bằng {row.get('material')}")
    if pd.notnull(row.get("weight_kg")): design.append(f"cân nặng {row.get('weight_kg')}kg")
    if pd.notnull(row.get("release_year")): design.append(f"Máy được ra mắt năm {int(row['release_year'])}")
    
    if design:
        parts.append("Thiết kế: " + ", ".join(design) + ".")

    # 5. port
    if pd.notnull(row.get("port")):
        parts.append(f"Máy gồm các cổng kết nối sau: {row.get('port')}.")
    #merge everything to become Full text
    page_content = " ".join(parts)

    metadata = {
            "id": str(row.get("id")),
            "brand": str(brand).lower(),
            "price_num": float(price / 1_000_000) if pd.notnull(price) else 0,
            "ram_gb": int(row.get("ram_info", 0)) if pd.notnull(row.get("ram_info")) else 0,
            "cpu": str(row.get("cpu_name", "")),
            "url": str(row.get("url_product", "")),
            "img": str(row.get("img_product", "")),
            "gpu_name": str(row.get("gpu_name", ""))
        }
    return Document(page_content=page_content, metadata=metadata)



In [ ]:
client = MongoClient("mongodb://localhost:27017/")
db = client["LaptopDataDB"]
collection = db["laptops_cleaned"]
data = pd.DataFrame(list(collection.find({}, {"_id": 0})))
documents = [create_laptop_document(item) for item in data]
embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert")
    # 4. Push to Chroma
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./vectorstore"
)
print("Finish Vector Database!")
                                   